# SurvCraft tutorial: Custom input and survival layers

This tutorial notebook shows how to implement custom input and survival layers in SurvCraft. 

It shows two examples:
1. A **custom input module** (a small CNN) and its **adapter**
2. A **custom survival module** (Log-Logistic distribution) and its **adapter**

The overall SurvCraft model is conceptually:

- `raw_params = input_module(X)`
- `preds = survival_module(mode, raw_params, times)`

Adapters are sklearn-style builders that create the torch modules with the right shapes.

## 0. Setup

> **Note**: The import paths below assume SurvCraft is installed as a package named `survcraft`.
If your local package name differs, adjust the imports accordingly.


In [1]:
import numpy as np
import torch

In [2]:
# SurvCraft imports (adjust if your package name differs)
from survcraft.adapters import SurvivalPredictor, BaseInputAdapter, BaseSurvivalAdapter
from survcraft import loss_modules
from survcraft.survival_modules import BaseSurvivalModule, PositiveParameter

## 1. Custom input module: a simple CNN

SurvCraft will call:

- `input_adapter.get_module(input_size=X.shape[1], output_size=P)`

Where `P` is the number of raw survival parameters produced by the input module, and must match
`survival_module.get_param_number()`.

If we represent images as `(N, C, H, W)`, then `X.shape[1] == C`, which is a natural `in_channels`
for a CNN. We use `AdaptiveAvgPool2d` so we **don't** need to hard-code `H` and `W`.


## FIXME integrare con sopra

Create a new input module and adapter to use with SurvivalPredictor

- a torch module whose constructor takes two integers as the first positional parameters, the input and output size.
- an adapter that extends the BaseInputAdapter class, has the module_class attribute with the torch module class as value and the other additional parameters to the module as attributes

The adapter is passed to the SurvivalPredictor and stores the additional parameters. In the fit method, the adapter will create an instance of the torch module using the number of features and the number of parameters required by the survival module as the input and output sizes.

In [3]:
class SimpleConvInputModule(torch.nn.Module):
    """
    Minimal CNN -> (batch, output_size) raw survival parameters.

    Expected input shape:
        X: (N, C, H, W)
    Where:
        input_size == C  (SurvCraft will pass X.shape[1])
    """

    def __init__(
        self,
        input_size: int,     # in_channels
        output_size: int,    # number of raw survival params
        channels=(16, 32),
        kernel_size=3,
        dropout=0.0,
    ):
        super().__init__()

        c1, c2 = channels
        pad = kernel_size // 2

        self.backbone = torch.nn.Sequential(
            torch.nn.Conv2d(input_size, c1, kernel_size=kernel_size, padding=pad),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),

            torch.nn.Conv2d(c1, c2, kernel_size=kernel_size, padding=pad),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),

            # (N, c2, h', w') -> (N, c2, 1, 1)
            torch.nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.head = torch.nn.Sequential(
            torch.nn.Flatten(),               # (N, c2)
            torch.nn.Dropout(dropout) if dropout > 0 else torch.nn.Identity(),
            torch.nn.Linear(c2, output_size), # (N, output_size)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x)
        x = self.head(x)
        return x


### 1.1 The corresponding adapter

Adapters are sklearn-style estimators that store hyperparameters and expose `get_module(...)`.
SurvCraft uses them to construct the final model inside `SurvivalPredictor`.


In [4]:
from dataclasses import dataclass

@dataclass
class SimpleConvInputAdapter(BaseInputAdapter):
    """Builds `SimpleConvInputModule` for SurvCraft."""
    module_class = SimpleConvInputModule

    channels: tuple = (16, 32)
    kernel_size: int = 3
    dropout: float = 0.0


### 1.2 Quick shape smoke test


In [5]:
# Suppose the survival module needs P=2 raw params (e.g., a 2-parameter distribution)
P = 2

adapter = SimpleConvInputAdapter(channels=(8, 16), dropout=0.1)
cnn = adapter.get_module(input_size=3, output_size=P)  # 3 channels (RGB)

X = torch.randn(4, 3, 64, 64)
out = cnn(X)

print("Output shape:", out.shape)


Output shape: torch.Size([4, 2])


## 2. Custom survival module: Log-Logistic distribution

We'll implement a Log-Logistic survival distribution with two **positive** parameters:

- `scale`  $(a > 0)$
- `shape`  $(b > 0)$

Survival:

$$ S(t) = \frac{1}{1 + (t/a)^b} $$

Density:
$$ f(t) = \frac{b}{a} (t/a)^{b-1} \cdot \frac{1}{(1 + (t/a)^b)^2} $$

Hazard:

$$ h(t) = \frac{b}{a}(t/a)^{b-1} \cdot \frac{1}{1+(t/a)^b} $$

SurvCraft's `BaseSurvivalModule` expects:
- `__init__` calls `super().__init__([Parameter(...), ...])`
- Methods like `survival`, `failure`, `density`, `hazard`, `expected_time`, `risk` return tensors with the expected shapes


In [6]:
class LogLogisticSurvivalModule(BaseSurvivalModule):
    name = "LogLogistic"

    def __init__(self, pp_func=PositiveParameter, *args, **kwargs):
        # Two positive params: scale and shape
        super().__init__([pp_func("scale"), pp_func("shape")], *args, **kwargs)

        # Useful constants/buffers
        self.register_buffer("zero", torch.tensor(0.0))

    def survival(self, params, times: torch.Tensor) -> torch.Tensor:
        a = params["scale"]
        b = params["shape"]
        t = torch.clamp(times, min=0.0)

        z = torch.pow(t / a, b)
        return 1.0 / (1.0 + z)

    def failure(self, params, times: torch.Tensor) -> torch.Tensor:
        return 1.0 - self.survival(params, times)

    def density(self, params, times: torch.Tensor) -> torch.Tensor:
        a = params["scale"]
        b = params["shape"]
        t = torch.clamp(times, min=0.0)

        # For t=0 and b<1, t^(b-1) can blow up; clamp for stability
        te = torch.clamp(t, min=self.epsilon.item())
        x = te / a
        xb = torch.pow(x, b)

        num = (b / a) * torch.pow(x, b - 1.0)
        den = torch.pow(1.0 + xb, 2.0)
        f = num / den

        # Define density at exactly t=0 as 0 to avoid NaNs
        return torch.where(t > 0.0, f, self.zero)

    def hazard(self, params, times: torch.Tensor) -> torch.Tensor:
        a = params["scale"]
        b = params["shape"]
        t = torch.clamp(times, min=0.0)

        te = torch.clamp(t, min=self.epsilon.item())
        x = te / a
        xb = torch.pow(x, b)

        h = (b / a) * torch.pow(x, b - 1.0) / (1.0 + xb)
        return torch.where(t > 0.0, h, self.zero)

    def median_time(self, params) -> torch.Tensor:
        # S(t)=0.5 -> t=a
        return params["scale"].squeeze(-1)

    def expected_time(self, params) -> torch.Tensor:
        # E[T] = a * (pi/b) * csc(pi/b) for b>1 else +inf
        a = params["scale"].squeeze(-1)
        b = params["shape"].squeeze(-1)

        x = torch.pi / b
        finite = b > 1.0

        et = a * x / torch.sin(x)  # x * csc(x)
        inf = torch.full_like(et, float("inf"))
        return torch.where(finite, et, inf)

    def risk(self, params) -> torch.Tensor:
        # Simple risk proxy (monotone in b/a):
        a = params["scale"].squeeze(-1)
        b = params["shape"].squeeze(-1)
        return b / a


### 2.1 The corresponding survival adapter


In [7]:
class LogLogisticSurvivalAdapter(BaseSurvivalAdapter):
    module_class = LogLogisticSurvivalModule


## 3. End-to-end example (toy data)

SurvCraft expects:

- `X`: numpy array
- `y`: structured array with fields `(event, time)` (scikit-survival style)

We'll create fake images and synthetic survival labels, then fit `SurvivalPredictor` using:

- our CNN input adapter
- our Log-Logistic survival adapter
- `BrierLoss` (uses model failure probabilities under the hood)


In [8]:
def make_toy_image_survival_data(n=256, seed=0):
    rng = np.random.default_rng(seed)

    # Fake images: (N, C, H, W)
    X = rng.normal(size=(n, 1, 32, 32)).astype(np.float32)

    # Create a signal from images (mean intensity)
    s = X.mean(axis=(1, 2, 3))

    # Synthetic times: smaller time for larger signal
    base = np.exp(-s) + 0.1
    time = (base + 0.2 * rng.random(n)).astype(np.float32)

    # Random censoring
    censor = rng.random(n) < 0.25
    event = (~censor).astype(bool)

    y = np.zeros(n, dtype=[("event", "?"), ("time", "f4")])
    y["event"] = event
    y["time"] = time
    return X, y

X, y = make_toy_image_survival_data()
print("X shape:", X.shape)
print("y dtype:", y.dtype)
print("First rows:", y[:3])


X shape: (256, 1, 32, 32)
y dtype: [('event', '?'), ('time', '<f4')]
First rows: [( True, 1.3289118) ( True, 1.2045577) (False, 1.2534732)]


### 3.1 Fit a model


In [9]:
model = SurvivalPredictor(
    input=SimpleConvInputAdapter(channels=(8, 16), dropout=0.1),
    survival=LogLogisticSurvivalAdapter(),
    loss=loss_modules.BrierLoss(),
    epochs=20,
    batch_size=128,
    learning_rate=1e-3,
    verbose=1,
)

model.fit(X, y)


Epoch 0, training loss = 0.2643924355506897
Epoch 1, training loss = 0.258083313703537
Epoch 2, training loss = 0.2541111707687378
Epoch 3, training loss = 0.25146499276161194
Epoch 4, training loss = 0.24921481311321259
Epoch 5, training loss = 0.24676546454429626
Epoch 6, training loss = 0.2463381588459015
Epoch 7, training loss = 0.24451783299446106
Epoch 8, training loss = 0.2433856576681137
Epoch 9, training loss = 0.2433781921863556
Epoch 10, training loss = 0.24139946699142456
Epoch 11, training loss = 0.24140310287475586
Epoch 12, training loss = 0.24229350686073303
Epoch 13, training loss = 0.24131472408771515
Epoch 14, training loss = 0.2410109043121338
Epoch 15, training loss = 0.24093855917453766
Epoch 16, training loss = 0.24110768735408783
Epoch 17, training loss = 0.24028179049491882
Epoch 18, training loss = 0.23909449577331543
Epoch 19, training loss = 0.23951679468154907
Final epoch 19, training loss = 0.23951679468154907


,input,"SimpleConvInp..., dropout=0.1)"
,survival,LogLogisticSurvivalAdapter()
,loss,BrierLoss()
,device,'cpu'
,verbose,1
,check_na,False
,batch_size,128
,learning_rate,0.001
,weight_decay,0.0
,epochs,20
,warm_start,False


### 3.2 Predict survival curves


In [10]:
times = np.linspace(0.0, 2.0, 50).astype(np.float32)
S = model.predict_survival(X[:5], times=times)  # expected shape (5, 50)

print("Survival output shape:", S.shape)
print("S[0, :5] =", S[0, :5])

Survival output shape: (5, 50)
S[0, :5] = [1.         0.9359022  0.9000959  0.87167656 0.8475457 ]


## 4. Troubleshooting

- **Shape mismatches**: ensure your input module outputs `(batch, P)` where `P = survival_module.get_param_number()`.
- **Wrong tensor rank**: the CNN expects `(N, C, H, W)`.
- **Numerical stability**: clamp times away from 0 in density/hazard if using powers like `t^(b-1)`.
